In [3]:
# step1_build_events_and_sequences.py
# Build Step 1 outputs:
# 1) business_lookup.parquet (business_id -> metadata incl. lat/lng)
# 2) user_visit_events.parquet (user_id, business_id, event_time, event_type + business meta)
# 3) user_sequences.json (user_id -> ordered list of business_ids)
# 4) user_sequences_meta.csv (user_id -> n_visits, start/end dates, unique businesses)
#
# Works with your 10k CSVs:
# - business_10k.csv
# - reviews_10k.csv
# - tips_10k.csv
# Optional:
# - user_ids_10k.txt (to filter to your sampled users)

import os
import json
import pandas as pd
from typing import Optional

# --------------------------
# CONFIG (edit these paths)
# --------------------------
BUSINESS_CSV = r"C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\business_10k_restaurants.csv"
REVIEWS_CSV  = r"C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\reviews_10k.csv"
TIPS_CSV     = r"C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\tips_10k.csv"
USER_IDS_TXT = r"C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\user_ids_10k.txt"  

OUT_DIR = r"C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\step1_outputs"
FILTER_TO_RESTAURANTS = True                   # keep only categories containing "Restaurants"
MIN_EVENTS_PER_USER = 5                        # keep users with at least N events (adjust for your experiments)


def read_user_ids(path: Optional[str]) -> Optional[set]:
    if not path or not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        ids = {line.strip() for line in f if line.strip()}
    return ids or None


def to_datetime(series: pd.Series) -> pd.Series:
    # Yelp dates are usually like "2018-06-01 12:34:56" or "2018-06-01"
    return pd.to_datetime(series, errors="coerce", utc=True)


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    user_ids = read_user_ids(USER_IDS_TXT)
    print("Loaded user_ids:", len(user_ids) if user_ids else "(no filter)")

    # --------------------------
    # 1) BUSINESS LOOKUP TABLE
    # --------------------------
    print("Loading business CSV...")
    biz = pd.read_csv(BUSINESS_CSV)

    # Normalize expected columns
    expected = {"business_id", "name", "latitude", "longitude", "city", "state", "categories"}
    missing = expected - set(biz.columns)
    if missing:
        raise ValueError(f"business_10k.csv missing columns: {missing}")

    biz["business_id"] = biz["business_id"].astype(str)

    # Optional: filter to restaurants (recommended for mobility)
    if FILTER_TO_RESTAURANTS:
        biz["categories"] = biz["categories"].fillna("").astype(str)
        biz = biz[biz["categories"].str.contains("Restaurants", case=False, na=False)].copy()

    # Keep only needed columns (+ rating signals if present)
    keep_cols = [c for c in ["business_id", "name", "latitude", "longitude", "city", "state", "categories",
                            "stars", "review_count", "is_open"] if c in biz.columns]
    biz_lookup = biz[keep_cols].drop_duplicates("business_id").copy()

    # Save business lookup
    biz_lookup_path = os.path.join(OUT_DIR, "business_lookup.parquet")
    biz_lookup.to_csv(biz_lookup_path, index=False)
    print("Wrote:", biz_lookup_path, "rows:", len(biz_lookup))

    # --------------------------
    # 2) USER VISIT EVENTS TABLE
    # --------------------------
    print("Loading reviews CSV...")
    reviews = pd.read_csv(REVIEWS_CSV)

    req_reviews = {"user_id", "business_id", "date"}
    missing_r = req_reviews - set(reviews.columns)
    if missing_r:
        raise ValueError(f"reviews_10k.csv missing columns: {missing_r}")

    reviews["user_id"] = reviews["user_id"].astype(str)
    reviews["business_id"] = reviews["business_id"].astype(str)
    reviews["event_time"] = to_datetime(reviews["date"])
    reviews["event_type"] = "review"

    # Keep minimal event fields (+ stars if present)
    review_keep = [c for c in ["user_id", "business_id", "event_time", "event_type", "stars"] if c in reviews.columns]
    reviews_ev = reviews[review_keep].copy()

    print("Loading tips CSV...")
    tips = pd.read_csv(TIPS_CSV)

    req_tips = {"user_id", "business_id", "date"}
    missing_t = req_tips - set(tips.columns)
    if missing_t:
        raise ValueError(f"tips_10k.csv missing columns: {missing_t}")

    tips["user_id"] = tips["user_id"].astype(str)
    tips["business_id"] = tips["business_id"].astype(str)
    tips["event_time"] = to_datetime(tips["date"])
    tips["event_type"] = "tip"

    tip_keep = [c for c in ["user_id", "business_id", "event_time", "event_type"] if c in tips.columns]
    tips_ev = tips[tip_keep].copy()

    # Combine events
    events = pd.concat([reviews_ev, tips_ev], ignore_index=True)

    # Optional: filter to sampled users
    if user_ids:
        events = events[events["user_id"].isin(user_ids)].copy()

    # Drop rows with missing timestamps
    events = events.dropna(subset=["event_time"]).copy()

    # Filter to businesses that exist in business_lookup (and restaurants, if enabled)
    biz_ids = set(biz_lookup["business_id"].unique())
    events = events[events["business_id"].isin(biz_ids)].copy()

    # Join business metadata (lat/lng etc.)
    events = events.merge(biz_lookup, on="business_id", how="left")

    # Sort for sequence creation
    events = events.sort_values(["user_id", "event_time"]).reset_index(drop=True)

    # Save events table
    events_path = os.path.join(OUT_DIR, "user_visit_events.parquet")
    events.to_csv(events_path, index=False)
    print("Wrote:", events_path, "rows:", len(events))

    # --------------------------
    # 3) USER SEQUENCES (for LOO eval later)
    # --------------------------
    # Create ordered list of business_ids per user
    grouped = events.groupby("user_id")["business_id"].apply(list)

    # Filter users with enough events
    grouped = grouped[grouped.apply(len) >= MIN_EVENTS_PER_USER]

    sequences = grouped.to_dict()

    seq_path = os.path.join(OUT_DIR, "user_sequences.json")
    with open(seq_path, "w", encoding="utf-8") as f:
        json.dump(sequences, f, ensure_ascii=False, indent=2)
    print("Wrote:", seq_path, "users:", len(sequences))

    # Optional: metadata per user (handy for debugging / selecting eval set)
    meta = events.groupby("user_id").agg(
        n_events=("business_id", "size"),
        n_unique_businesses=("business_id", "nunique"),
        start_time=("event_time", "min"),
        end_time=("event_time", "max"),
    ).reset_index()

    meta = meta[meta["n_events"] >= MIN_EVENTS_PER_USER].copy()
    meta_path = os.path.join(OUT_DIR, "user_sequences_meta.csv")
    meta.to_csv(meta_path, index=False)
    print("Wrote:", meta_path, "rows:", len(meta))

    # --------------------------
    # Quick sanity prints
    # --------------------------
    print("\nSanity check:")
    print("  Example user:", meta.iloc[0]["user_id"] if len(meta) else "(none)")
    if len(meta):
        uid0 = meta.iloc[0]["user_id"]
        print("  Events for example user:", meta.iloc[0]["n_events"])
        print("  First 5 businesses:", sequences[uid0][:5])

    print("\nDone ✅")


if __name__ == "__main__":
    main()

Loaded user_ids: 10000
Loading business CSV...
Wrote: C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\step1_outputs\business_lookup.parquet rows: 14027
Loading reviews CSV...
Loading tips CSV...
Wrote: C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\step1_outputs\user_visit_events.parquet rows: 27732
Wrote: C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\step1_outputs\user_sequences.json users: 1159
Wrote: C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\step1_outputs\user_sequences_meta.csv rows: 1159

Sanity check:
  Example user: -0MIp6WKJ8QvGnYZQ5ETyg
  Events for example user: 19
  First 5 businesses: ['UTgUhbpM5hRVDB8Nu7-lpw', 'jwpZzqoPFLuHE-FwLp41cQ', 'uXziexv0fsfRMw1hEdifow', 'E7pGpkLw7i73oeuf8tvJog', '1wKbk-FtJBBidd1k8s09DA']

Done ✅


In [6]:
import json
from collections import Counter
import numpy as np

seq_path = r"C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\step1_outputs\\user_sequences.json"
with open(seq_path, "r", encoding="utf-8") as f:
    user_seqs = json.load(f)

lengths = np.array([len(v) for v in user_seqs.values()])

print("Users:", len(lengths))
print("Min/Median/Mean/Max:", lengths.min(), np.median(lengths), lengths.mean().round(2), lengths.max())
print("Users >= 5:", (lengths >= 5).sum())
print("Users >= 10:", (lengths >= 10).sum())

# distribution buckets
buckets = Counter()
for L in lengths:
    if L == 1: buckets["1"] += 1
    elif L <= 3: buckets["2-3"] += 1
    elif L <= 5: buckets["4-5"] += 1
    elif L <= 10: buckets["6-10"] += 1
    else: buckets["11+"] += 1

print("Buckets:", buckets)

Users: 1159
Min/Median/Mean/Max: 5 8.0 15.3 315
Users >= 5: 1159
Users >= 10: 490
Buckets: Counter({'6-10': 501, '11+': 433, '4-5': 225})


In [7]:
import pandas as pd

meta_path = r"C:\\Users\\lebro\\OneDrive - Nanyang Technological University\\Github\\YelpFYP\\step1_outputs\\user_sequences_meta.csv" 
meta = pd.read_csv(meta_path)

print(meta.columns)
print(meta["n_events"].describe())
print("Users >= 5:", (meta["n_events"] >= 5).sum())
print("Users >= 10:", (meta["n_events"] >= 10).sum())

Index(['user_id', 'n_events', 'n_unique_businesses', 'start_time', 'end_time'], dtype='object')
count    1159.000000
mean       15.304573
std        22.643736
min         5.000000
25%         6.000000
50%         8.000000
75%        15.000000
max       315.000000
Name: n_events, dtype: float64
Users >= 5: 1159
Users >= 10: 490
